In [1]:
import numpy as np

Чтение кода

a = np.array([2, 4, 6]); print(a / 2, a // 4)

Ответ: [1. 2. 3.] [0 1 1]

In [2]:
a = np.array([2, 4, 6]); print(a / 2, a // 4)

[1. 2. 3.] [0 1 1]


m = np.ones((3, 4)); print(m.sum(axis=1))
Ответ: [4. 4. 4.]

In [3]:
m = np.ones((3, 4)); print(m.sum(axis=1))

[4. 4. 4.]


print(np.arange(3) + np.arange(3).reshape(3, 1))
Ответ: бут матричное присоединение 3х3

In [4]:
print(np.arange(3) + np.arange(3).reshape(3, 1))

[[0 1 2]
 [1 2 3]
 [2 3 4]]


x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
Ответ будет 1+2+3, так как нужна сумма корней списка, значит 6.0

In [5]:
x = np.array([1, 4, 9])
print(np.sqrt(x).sum())

6.0


print((np.arange(1, 5) ** 2).mean())
Ответ будет среднеарифметическое суммы квадратов 1 до 4, то есть 7.5


In [6]:
print((np.arange(1, 5) ** 2).mean())

7.5


In [7]:
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

Saving orders.csv to orders.csv
orders.csv на месте: True


In [8]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [9]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


База TASK 1

In [10]:
print(clean.sum())
rows_np = np.array(rows)
print(len(rows_np[clean]))

print(month)

clean_sep = clean & (month == '2026-09')
clean_oct = clean & (month == '2026-10')

total_sep = amount[clean_sep].sum()
total_oct = amount[clean_oct].sum()
changes = ((total_oct - total_sep) / total_sep) * 100


print(total_sep)
print(total_oct)
print(changes)



4739
4739
['2026-09' '2026-09' '2026-09' ... '2026-10' '2026-10' '2026-10']
35243410.0
34419520.0
-2.337713632137185


База TASK 2

In [11]:
month_unique = np.unique(month[clean])
cat_unique = np.unique(cat[clean])

table = np.array([[amount[clean & (cat == c) & (month == m)].sum() for m in month_unique] for c in cat_unique])

pct_change = ((table[:, 1] - table[:, 0]) / table[:, 0]) * 100
worst_change = pct_change.argmin()

print(pct_change)
print(table)
print(cat_unique[table.sum(axis=1).argmax()])
print(cat_unique[worst_change], np.abs(pct_change.min()))

[ 2.56236331  7.09753121  2.60594803 -7.2146044  -2.49524058]
[[ 4997730.  5125790.]
 [ 1065300.  1140910.]
 [ 2610950.  2678990.]
 [ 9167100.  8505730.]
 [17402330. 16968100.]]
Электроника
Одежда 7.214604400519248


База TASK 3

In [12]:
table_2 = np.array([[len(amount[clean & (cat == c) & (month == month_unique[1])])] for c in cat_unique])


print(table_2)
avg_check = (table[:, 1].reshape(-1, 1) / table_2)

print(avg_check)

print(cat_unique[avg_check.argmax()], avg_check.max())

[[476]
 [299]
 [435]
 [652]
 [505]]
[[10768.46638655]
 [ 3815.75250836]
 [ 6158.59770115]
 [13045.59815951]
 [33600.1980198 ]]
Электроника 33600.19801980198


Стандарт TASK 4

In [29]:
city_unique = np.unique(city)
table_city = np.array([[amount[clean & (month == m) & (city == c)].sum() for m in month_unique] for c in city_unique])


print(f'{'Gorod':<10} | {'September':>10} | {'October':>10}')
print('-' * 50)
for c, t in zip(city_unique, table_city):
  print(f'{c:<10} | {t[0]:>10} | {t[1]:>10}')




total_sep = table_city[:, 0].sum(axis=0)
total_oct = table_city[:, 1].sum(axis=0)
print(total_sep, total_oct)

table_city_prc = (table_city / [total_sep, total_oct])

print(table_city_prc)
print(table_city_prc.sum(axis=0))

Gorod      |  September |    October
--------------------------------------------------
Актобе     |  2576960.0 |  2540420.0
Алматы     | 14501070.0 | 14643020.0
Астана     |  9034220.0 |  8850530.0
Атырау     |   923990.0 |   969860.0
Караганда  |  3648520.0 |  3371680.0
Шымкент    |  4558650.0 |  4044010.0
35243410.0 34419520.0
[[0.07311892 0.07380754]
 [0.41145479 0.42542778]
 [0.25633785 0.25713694]
 [0.02621738 0.02817762]
 [0.10352347 0.09795837]
 [0.12934759 0.11749176]]
[1. 1.]


Стандарт TASK 5

In [60]:
city_drop_money = table_city[:, 0] - table_city[:, 1]
total_drop_money = (total_sep - total_oct).sum()

city_cont = (city_drop_money / total_drop_money) * 100
worst_city_idx = city_drop_money.argmax()

print(city_unique[worst_city_idx], city_cont[worst_city_idx])

#В Шымкенте скорее всего одежда

[  4.43505808 -17.22924177  22.29545206  -5.56749081  33.60157303
  62.46464941]
Шымкент 62.46464940708104


Стандарт TASK 6



In [62]:
share = table / table.sum(axis=1, keepdims=True)
# keepdims=True